# Model Inference - IEEE-CIS Fraud Detection



## საუკეთესო მოდელი
- **მოდელი:** XGBoost Classifier
- **Validation AUC:** 0.9003
- **Kaggle Public LB:** 0.9176
- **Kaggle Private LB:** 0.8793
- **Model Registry:** `XGB_Fraud_Detection`

## შედარება ყველა მოდელის
| Model | Public LB | Private LB |
|---|---|---|
| Logistic Regression | 0.861 | 0.830 |
| Random Forest | 0.904 | 0.872 |
| **XGBoost**  | **0.918** | **0.879** |

## Test Set Pipeline
1. Load Pipeline from MLflow Model Registry
2. Load test data (raw, preprocessed არ არის)
3. Apply Feature Engineering (იგივე რაც train-ზე)
4. `pipeline.predict_proba(test)` — Pipeline ყველაფერს თვითონ აკეთებს
5. Submission file generation

In [1]:
# ============================================================
# Setup: Libraries
# ============================================================
!pip install dagshub -q
!pip install mlflow -q
!pip install category_encoders -q
!pip install xgboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [2]:
import dagshub
dagshub.init(repo_owner='vasiliDandurishvili', repo_name='ML_Fraud_Detection', mlflow=True)

import mlflow
print(" MLflow connected to DagsHub")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=33a806bf-5ebc-4331-8ea0-9932b9650456&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=17f63fa5485e67ed1085d8b52163c6ff377a086be5a8ce74149ce98982ba88d8




Output()

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt

# Pipeline-ისთვის (custom transformer-ი reload-ისას სჭირდება)
from sklearn.base import BaseEstimator, TransformerMixin



## Custom Transformer-ის დარეგისტრირება

 **მნიშვნელოვანი:** Pipeline-ის custom class-ები (StringImputer) Inference-ში ისევ უნდა განვსაზღვროთ, რომ unpickling-მა იპოვოს class-ი.

In [ ]:
# Custom transformer ცდა Pipeline-ის შიგნით გვქონდა
# Pickling/unpickling-ისთვის class definition აუცილებელია

class StringImputer(BaseEstimator, TransformerMixin):
    """Categorical NaN → 'missing' string."""
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)
        return X.astype(str).fillna('missing').replace('nan', 'missing')


## 1. Load Pipeline from Model Registry

In [ ]:
# ============================================================
# ჩამოვტვირთოთ საუკეთესო Pipeline DagsHub Model Registry-დან
# ============================================================
import mlflow.sklearn

MODEL_NAME = "XGB_Fraud_Detection"
MODEL_VERSION = "latest"  # ან კონკრეტული ვერსია, მაგ. "1"

print(f"Loading model: {MODEL_NAME} (version: {MODEL_VERSION})...")

model_uri = f"models:/{MODEL_NAME}/{MODEL_VERSION}"
pipeline = mlflow.sklearn.load_model(model_uri)


print(f"\nPipeline structure:")
print(pipeline)

## 2. Load Test Data (Raw)

Test-ი ატვირთული ფორმით - **არანაირი preprocessing არ ვაკეთებთ ხელით**, Pipeline-ი ყველაფერს თვითონ აკეთებს.

In [ ]:
# ============================================================
# Load Test Data
# ============================================================

# dataset-ის path Kaggle-ზე
DATA_PATH = '/kaggle/input/datasets/vasilidandurishvili/dataset/'

print("Loading test data...")
test_tx = pd.read_csv(DATA_PATH + 'test_transaction.csv')
test_id = pd.read_csv(DATA_PATH + 'test_identity.csv')

# test_identity-ის სვეტებში id-XX აქვს ნაცვლად id_XX-სი
test_id.columns = [c.replace('-', '_') for c in test_id.columns]

# Merge
test = test_tx.merge(test_id, on='TransactionID', how='left')

print(f"Test shape: {test.shape}")

del test_tx, test_id
gc.collect()

In [ ]:
def reduce_mem_usage(df):
    """Memory reduction trick - downcasts numeric types."""
    start_mem = df.memory_usage().sum() / 1024**2
    
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            try:
                c_min = df[col].min()
                c_max = df[col].max()
                if str(col_type)[:3] == 'int':
                    if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        df[col] = df[col].astype(np.int8)
                    elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                        df[col] = df[col].astype(np.int16)
                    elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                        df[col] = df[col].astype(np.int32)
                else:
                    if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                        df[col] = df[col].astype(np.float32)
            except:
                pass
    
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'  {start_mem:.1f} MB → {end_mem:.1f} MB ({100*(start_mem-end_mem)/start_mem:.1f}% reduction)')
    return df

print("Reducing test memory...")
test = reduce_mem_usage(test)

## 3. Feature Engineering

 Pipeline-ი არ აკეთებს Feature Engineering-ს — ის მხოლოდ Cleaning-ის და FS-ის შემდეგ გამოიყენება. ე.ი. იგივე FE რასაც training-ზე ვაკეთებდით, აქაც უნდა გავიმეოროთ.

In [ ]:
# ============================================================
# Feature Engineering — იგივე 5 feature რაც train-ზე
# ============================================================

print("Applying Feature Engineering...")

# 1. TransactionAmt log
test['TransactionAmt_log'] = np.log1p(test['TransactionAmt'])

# 2. TransactionAmt decimal
test['TransactionAmt_decimal'] = test['TransactionAmt'] - test['TransactionAmt'].astype(int)

# 3. Time features
test['hour'] = (test['TransactionDT'] / 3600) % 24
test['day'] = (test['TransactionDT'] / (3600 * 24)) % 7

# 4. Email domain bin
for col in ['P_emaildomain', 'R_emaildomain']:
    if col in test.columns:
        test[col + '_bin'] = test[col].astype(str).str.split('.').str[0]

print(f" Test shape: {test.shape}")

## 4. Test Predictions

Pipeline თავად აკეთებს:
- Categorical Frequency Encoding 
- XGBoost Native NaN handling (numerical)
- Prediction 

In [ ]:
# ============================================================
# Test Predictions - Pipeline ყველაფერს თვითონ აკეთებს
# ============================================================

# Pipeline-ი რომელ features-ს ელის
preprocessor = pipeline.named_steps['preprocessor']
required_features = []
for name, _, cols in preprocessor.transformers:
    if cols != 'drop' and isinstance(cols, list):
        required_features.extend(cols)

print(f"Pipeline expects {len(required_features)} features")

#  ყველა საჭირო feature არსებობობდეს test-ში
missing = [c for c in required_features if c not in test.columns]
if missing:
    print(f" Missing features in test: {missing[:10]}")
    # თუ რამე feature აკლია (cleaning-ით drop-ვდა train-ში), დავამატოთ NaN-ით
    for col in missing:
        test[col] = np.nan
    print(f"  Added {len(missing)} missing features as NaN")
else:
    print("All required features present")

X_test = test[required_features]
print(f"X_test shape: {X_test.shape}")

# Predict!
print("\nPredicting...")
test_pred = pipeline.predict_proba(X_test)[:, 1]

print(f"\n Predictions complete")
print(f"  Predictions: {len(test_pred):,}")
print(f"  Mean: {test_pred.mean():.4f}")
print(f"  Min: {test_pred.min():.6f}, Max: {test_pred.max():.6f}")

## 5. Submission File

In [ ]:
# ============================================================
# Submission File Generation
# ============================================================

submission = pd.DataFrame({
    'TransactionID': test['TransactionID'],
    'isFraud': test_pred
})

submission_path = '/kaggle/working/submission_inference.csv'
submission.to_csv(submission_path, index=False)

print(f" Submission saved: {submission_path}")
print(f" Rows: {len(submission):,}")
print(f"\nPrediction distribution:")
print(submission['isFraud'].describe())

# Visualize
plt.figure(figsize=(12, 4))
plt.hist(submission['isFraud'], bins=100, color='steelblue', edgecolor='black')
plt.title('XGBoost Inference - Test Predictions Distribution')
plt.xlabel('Predicted P(fraud)')
plt.ylabel('Count')
plt.axvline(x=0.5, color='red', linestyle='--', label='Threshold = 0.5')
plt.legend()
plt.tight_layout()
plt.show()

print("  Upload submission_inference.csv to Kaggle competition")

In [ ]:
import os
print("Output files:")
for f in os.listdir('/kaggle/working/'):
    file_path = f'/kaggle/working/{f}'
    size = os.path.getsize(file_path) / 1024  # KB
    print(f"  {f} ({size:.1f} KB)")